# 01 — Model-independent conformal classification baseline

**Goal.** Develop and validate the conformal-prediction machinery before connecting it to ERGO.

The notebook interface is deliberately simple:

- `probs`: NumPy array with shape `[N, K]`, initially `K=10`
- `labels`: integer NumPy array with shape `[N]`

We start with split conformal prediction at $\alpha=0.1$ (90% nominal marginal coverage), compare a simple probability/nonconformity score with APS, then evaluate marginal and class-specific reliability. The code is written so stored classifier outputs can later replace the synthetic data without changing the conformal machinery.

Reference implementation/template: Angelopoulos & Bates, `aangelopoulos/conformal-prediction`, especially the ImageNet classification/APS notebooks. The implementation below is kept explicit so every statistical step can be inspected and tested.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Optional

SEED = 42
ALPHA = 0.10
rng = np.random.default_rng(SEED)


## 1. Finite-sample conformal quantile

For `n` calibration examples, split conformal uses the corrected empirical level

$$q_{\mathrm{level}} = \frac{\lceil (n+1)(1-\alpha)\rceil}{n},$$

clipped at 1. Using NumPy's `method="higher"` selects an observed calibration score rather than interpolating between scores.


In [ ]:
def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    scores = np.asarray(scores, dtype=float)
    if scores.ndim != 1 or len(scores) == 0:
        raise ValueError("scores must be a non-empty 1D array")
    n = len(scores)
    level = min(1.0, np.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, level, method="higher"))


## 2. Synthetic classifier outputs

This is **not** an experimental result. It is a software/statistical sanity check. We generate ten-class probability vectors with deliberately unequal class difficulty so that marginal coverage can be compared with class-wise coverage before ERGO is available.


In [ ]:
def make_synthetic_probabilities(n=12000, k=10, seed=42):
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, k, size=n)
    logits = rng.normal(0, 1.0, size=(n, k))

    # Different true-class boosts create easy and difficult classes.
    boosts = np.linspace(3.5, 1.0, k)
    logits[np.arange(n), labels] += boosts[labels]

    logits -= logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return probs, labels

probs, labels = make_synthetic_probabilities()
assert probs.shape == (12000, 10)
assert labels.shape == (12000,)
assert np.allclose(probs.sum(axis=1), 1.0)

perm = rng.permutation(len(labels))
cut = len(labels) // 2
cal_idx, test_idx = perm[:cut], perm[cut:]
cal_probs, cal_labels = probs[cal_idx], labels[cal_idx]
test_probs, test_labels = probs[test_idx], labels[test_idx]

print(cal_probs.shape, test_probs.shape)


## 3. Baseline: label-wise probability score (LAC-style)

Use

$$s(x,y)=1-\hat p_y(x).$$

Calibration scores are evaluated only at the true calibration labels. A candidate test label $y$ is included when its score is no larger than the calibrated threshold $\hat q$.


In [ ]:
@dataclass
class PredictionSetResult:
    sets: np.ndarray          # bool array [N, K]
    threshold: object
    method: str


def fit_lac(cal_probs, cal_labels, alpha=0.1):
    scores = 1.0 - cal_probs[np.arange(len(cal_labels)), cal_labels]
    return conformal_quantile(scores, alpha)


def predict_lac(probs, qhat):
    return (1.0 - probs) <= qhat

q_lac = fit_lac(cal_probs, cal_labels, ALPHA)
lac_sets = predict_lac(test_probs, q_lac)
print(f"LAC threshold: {q_lac:.4f}")


## 4. APS: Adaptive Prediction Sets

APS ranks the class probabilities from largest to smallest and scores a candidate class by the cumulative probability mass up to that class. We use the randomized APS score

$$s(x,y;u)=\sum_{j:\,\hat p_j(x)>\hat p_y(x)}\hat p_j(x)+u\hat p_y(x), \qquad u\sim U(0,1).$$

Randomization handles the boundary class and matches the generalized inverse-quantile construction underlying APS. Fixing the random seed makes experiments reproducible. For a deterministic diagnostic, set `randomized=False`, which uses the full cumulative mass through the candidate class.


In [ ]:
def aps_scores(probs, labels, rng=None, randomized=True):
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    order = np.argsort(-probs, axis=1)
    sorted_probs = np.take_along_axis(probs, order, axis=1)
    cumsum = np.cumsum(sorted_probs, axis=1)
    ranks = np.argsort(order, axis=1)[np.arange(len(labels)), labels]
    p_y = probs[np.arange(len(labels)), labels]
    before = cumsum[np.arange(len(labels)), ranks] - p_y

    if randomized:
        if rng is None:
            rng = np.random.default_rng()
        u = rng.random(len(labels))
        return before + u * p_y
    return before + p_y


def fit_aps(cal_probs, cal_labels, alpha=0.1, seed=42, randomized=True):
    local_rng = np.random.default_rng(seed)
    scores = aps_scores(cal_probs, cal_labels, local_rng, randomized=randomized)
    return conformal_quantile(scores, alpha)


def predict_aps(probs, qhat, seed=43, randomized=True):
    probs = np.asarray(probs, dtype=float)
    n, k = probs.shape
    order = np.argsort(-probs, axis=1)
    sorted_probs = np.take_along_axis(probs, order, axis=1)
    cumsum = np.cumsum(sorted_probs, axis=1)
    before = cumsum - sorted_probs

    if randomized:
        local_rng = np.random.default_rng(seed)
        u = local_rng.random((n, k))
        sorted_scores = before + u * sorted_probs
    else:
        sorted_scores = cumsum

    include_sorted = sorted_scores <= qhat
    sets = np.zeros((n, k), dtype=bool)
    np.put_along_axis(sets, order, include_sorted, axis=1)
    return sets

q_aps = fit_aps(cal_probs, cal_labels, ALPHA, seed=SEED)
aps_sets = predict_aps(test_probs, q_aps, seed=SEED + 1)
print(f"APS threshold: {q_aps:.4f}")


## 5. Evaluation framework

We evaluate more than overall coverage. The central project question requires seeing whether nominal marginal reliability conceals poor reliability for particular conditions.


In [ ]:
def evaluate_prediction_sets(sets, labels, class_names=None):
    sets = np.asarray(sets, dtype=bool)
    labels = np.asarray(labels, dtype=int)
    n, k = sets.shape
    if class_names is None:
        class_names = [f"Class {i}" for i in range(k)]

    covered = sets[np.arange(n), labels]
    sizes = sets.sum(axis=1)

    summary = {
        "n": n,
        "marginal_coverage": covered.mean(),
        "mean_set_size": sizes.mean(),
        "median_set_size": np.median(sizes),
        "singleton_fraction": np.mean(sizes == 1),
        "multi_class_fraction": np.mean(sizes > 1),
        "empty_fraction": np.mean(sizes == 0),
    }

    rows = []
    for c, name in enumerate(class_names):
        mask = labels == c
        rows.append({
            "class": c,
            "name": name,
            "n": int(mask.sum()),
            "coverage": covered[mask].mean() if mask.any() else np.nan,
            "mean_set_size": sizes[mask].mean() if mask.any() else np.nan,
        })
    return summary, pd.DataFrame(rows), covered, sizes


def compare_methods(method_sets, labels):
    rows = []
    details = {}
    for name, sets in method_sets.items():
        summary, by_class, covered, sizes = evaluate_prediction_sets(sets, labels)
        rows.append({"method": name, **summary})
        details[name] = by_class
    return pd.DataFrame(rows), details

comparison, class_details = compare_methods(
    {"LAC": lac_sets, "APS": aps_sets}, test_labels
)
comparison


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(10)
width = 0.38
ax.bar(x - width/2, class_details["LAC"]["coverage"], width, label="LAC")
ax.bar(x + width/2, class_details["APS"]["coverage"], width, label="APS")
ax.axhline(1 - ALPHA, linestyle="--", label="Nominal coverage")
ax.set_ylim(0, 1.02)
ax.set_xlabel("True class")
ax.set_ylabel("Empirical coverage")
ax.set_title("Marginal validity does not imply equal class-wise coverage")
ax.set_xticks(x)
ax.legend()
plt.show()


## 6. Class-conditional conformal calibration

For each true class $k$, estimate its own threshold $\hat q^{(k)}$ from calibration examples with $Y=k$. At test time, candidate label $y$ is tested against $\hat q^{(y)}$. This targets class-balanced coverage rather than only global marginal coverage.

We first implement this for the simple probability score so that the effect of changing the calibration target is isolated from the effect of changing the conformity score.


In [ ]:
def fit_class_conditional_lac(cal_probs, cal_labels, alpha=0.1):
    k = cal_probs.shape[1]
    q = np.empty(k)
    for c in range(k):
        mask = cal_labels == c
        if mask.sum() == 0:
            raise ValueError(f"No calibration examples for class {c}")
        scores_c = 1.0 - cal_probs[mask, c]
        q[c] = conformal_quantile(scores_c, alpha)
    return q


def predict_class_conditional_lac(probs, q_by_class):
    return (1.0 - probs) <= np.asarray(q_by_class)[None, :]

q_cc = fit_class_conditional_lac(cal_probs, cal_labels, ALPHA)
cc_sets = predict_class_conditional_lac(test_probs, q_cc)

comparison_cc, details_cc = compare_methods(
    {"Marginal LAC": lac_sets, "Class-conditional LAC": cc_sets},
    test_labels,
)
comparison_cc


## 7. Reproducibility experiment

A single calibration split is not enough for the IP. Repeating the split lets us quantify how much coverage and set size move solely because a different calibration sample was selected.


In [ ]:
def repeated_lac_experiment(probs, labels, alpha=0.1, seeds=range(30), cal_fraction=0.5):
    rows = []
    n = len(labels)
    n_cal = int(n * cal_fraction)
    for seed in seeds:
        local_rng = np.random.default_rng(seed)
        idx = local_rng.permutation(n)
        cal, test = idx[:n_cal], idx[n_cal:]
        q = fit_lac(probs[cal], labels[cal], alpha)
        sets = predict_lac(probs[test], q)
        summary, by_class, _, _ = evaluate_prediction_sets(sets, labels[test])
        row = {"seed": seed, "qhat": q, **summary}
        for c, coverage in zip(by_class["class"], by_class["coverage"]):
            row[f"coverage_class_{c}"] = coverage
        rows.append(row)
    return pd.DataFrame(rows)

repeats = repeated_lac_experiment(probs, labels, ALPHA)
repeats[["marginal_coverage", "mean_set_size"]].describe()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
coverage_cols = [f"coverage_class_{c}" for c in range(10)]
ax.boxplot([repeats[c].dropna() for c in coverage_cols], labels=range(10))
ax.axhline(1 - ALPHA, linestyle="--")
ax.set_xlabel("True class")
ax.set_ylabel("Coverage across calibration splits")
ax.set_title("Reproducibility of class-wise coverage")
plt.show()


## 8. ERGO adapter — later

When classifier outputs become available, the synthetic block above should be replaced by a small loader only. The conformal and evaluation code should remain unchanged.

Expected minimum format:

```python
probs = np.load("ergo_probs.npy")    # [N, 10], rows sum to 1
labels = np.load("ergo_labels.npy")  # [N], integer class labels
```

Before running any experiment we must establish whether observations are independent at image, study, or patient level. Calibration/test splitting must respect the appropriate unit to avoid leakage.

### Next experiments

1. Verify baseline marginal coverage over many random splits.
2. Compare LAC and APS efficiency and class-wise coverage.
3. Compare marginal and class-conditional calibration.
4. Sweep calibration-set size and $\alpha$.
5. Add confidence intervals / Monte Carlo summaries.
6. Only then connect ERGO outputs.
7. Stretch: RAPS, CV+/jackknife+, subgroup coverage and distribution shift.
